# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

data_path = "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

df_model = df.dropna(subset=features + ["client_id"]).copy()

scaler = StandardScaler()
X = scaler.fit_transform(df_model[features])

# Same number of clusters as Week 5
k = 2

model = KMeans(n_clusters=k, random_state=42, n_init=10)
df_model["cluster"] = model.fit_predict(X)

cluster_profile = (
    df_model.groupby("cluster")[features]
    .median()
    .round(2)
)

cluster_profile


action_map = {
    0: "review_and_refresh",
    1: "monitor",
}

reason_map = {
    0: "lower_visibility_and_engagement_pattern",
    1: "stronger_performance_pattern",
}

df_model["recommended_action"] = df_model["cluster"].map(action_map)
df_model["reason_code"] = df_model["cluster"].map(reason_map)

df_model["action_priority"] = df_model["recommended_action"].map({
    "review_and_refresh": 1,
    "monitor": 2,
})

queue = df_model.sort_values(
    ["action_priority", "impressions_90d"],
    ascending=[True, False]
).copy()

queue["rank"] = range(1, len(queue) + 1)

queue[
    [
        "rank",
        "content_id",
        "client_id",
        "cluster",
        "recommended_action",
        "reason_code",
        "impressions_90d",
        "avg_position",
        "ctr",
        "engagement_rate",
        "word_count",
        "days_since_last_update",
    ]
].head(20)

,rank,content_id,client_id,cluster,recommended_action,reason_code,impressions_90d,avg_position,ctr,engagement_rate,word_count,days_since_last_update
26336,1,content_5b6bb729c889,client_19581e27de,0,review_and_refresh,lower_visibility_and_engagement_pattern,69104,8.6,0.07,2.90,2218.0,20
14226,2,content_65d9331f55fb,client_4e07408562,0,review_and_refresh,lower_visibility_and_engagement_pattern,65686,7.8,0.02,3.70,3161.0,7
13631,3,content_d274ac4158ef,client_4e07408562,0,review_and_refresh,lower_visibility_and_engagement_pattern,65138,6.8,0.01,4.00,1428.0,26
5338,4,content_d7175187ff12,client_4e07408562,0,review_and_refresh,lower_visibility_and_engagement_pattern,60644,4.3,0.12,6.78,2627.0,14
6406,5,content_60a90d0ba16a,client_f369cb89fc,0,review_and_refresh,lower_visibility_and_engagement_pattern,57709,1.8,0.27,2.97,2471.0,20
24866,6,content_e5f459e737b7,client_f369cb89fc,0,review_and_refresh,lower_visibility_and_engagement_pattern,56363,5.9,0.01,0.00,2916.0,20
29660,7,content_6926821766df,client_f369cb89fc,0,review_and_refresh,lower_visibility_and_engagement_pattern,56082,4.4,0.20,3.96,2615.0,20
8329,8,content_d55b83aff7c7,client_f369cb89fc,0,review_and_refresh,lower_visibility_and_engagement_pattern,54704,6.8,0.06,5.71,2892.0,8
29464,9,content_548238274473,client_7f2253d7e2,0,review_and_refresh,lower_visibility_and_engagement_pattern,53903,40.3,0.02,9.09,3009.0,20
28941,10,content_717e10c0118b,client_349c41201b,0,review_and_refresh,lower_visibility_and_engagement_pattern,52148,13.3,0.14,2.36,2831.0,20


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
print("""
I would use the clusters to group pages with similar performance patterns.

The main use is to help decide which pages should be reviewed first.
The clusters can support content review, but they are not final decisions.

The result is based on the features in the dataset, so it can change when
the data or content mix changes.
""")


I would use the clusters to group pages with similar performance patterns.

The main use is to help decide which pages should be reviewed first.
The clusters can support content review, but they are not final decisions.

The result is based on the features in the dataset, so it can change when
the data or content mix changes.



## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
print("""
A person should review a page before making a content change.

The reviewer should check the page, its search intent, recent changes,
and whether the data still represents the current page.

The following should not be automated:
- deleting content
- changing the main search intent
- making major SEO changes
- publishing changes without review
- treating a cluster as a quality label
""")


A person should review a page before making a content change.

The reviewer should check the page, its search intent, recent changes,
and whether the data still represents the current page.

The following should not be automated:
- deleting content
- changing the main search intent
- making major SEO changes
- publishing changes without review
- treating a cluster as a quality label



## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
print("""
I would check the clusters again when the data changes a lot.

Useful triggers are:
- a large change in the number of pages
- a large change in the content mix
- important changes in traffic or engagement
- cluster sizes changing a lot
- cluster profiles becoming different from the previous run

If these changes happen, I would rerun the clustering and review the actions.
""")


I would check the clusters again when the data changes a lot.

Useful triggers are:
- a large change in the number of pages
- a large change in the content mix
- important changes in traffic or engagement
- cluster sizes changing a lot
- cluster profiles becoming different from the previous run

If these changes happen, I would rerun the clustering and review the actions.



## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.